In [2]:
import os
import numpy as np  # type: ignore
import matplotlib.pyplot as plt  # type: ignore
from matplotlib.patches import Arc  # type: ignore
from datetime import datetime

RESULT_DIR = os.path.join(os.getcwd(), "result")
os.makedirs(RESULT_DIR, exist_ok=True)
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# ─────────────────────────────────────────────
# STEP 1: Two arrows on one flat page (2D): the lever and the force
#
# What we have: a pivot O at the origin, a lever arrow r = (p, q) m from O to its tip,
#               and a force arrow F = (F_p, F_q) N that starts at the tip.
# What we want: to see the two angles phi_r and phi_F (both measured from the p axis)
#               and the gap between them, theta = phi_F - phi_r.
# Why: T1 gave tau = rho f sin(theta). We need theta from coordinates, and theta is the
#      difference of the two arrow directions. The dashed line at the tip is the p direction
#      copied to the tip, so phi_F is measured from the same ruler as phi_r.
#
# Formulas (N*m):
#   tau = rho f sin(phi_F - phi_r) = p F_q - q F_p
# ─────────────────────────────────────────────


def draw(ax, r, F, scale, title):
    p, q = r
    Fp, Fq = F
    rho = np.hypot(p, q)  # m
    f = np.hypot(Fp, Fq)  # N
    phi_r = np.degrees(np.arctan2(q, p))
    phi_F = np.degrees(np.arctan2(Fq, Fp))
    tau = p * Fq - q * Fp  # N*m

    # axes
    ax.annotate("", xy=(1.6, 0), xytext=(-0.2, 0), arrowprops=dict(arrowstyle="->", color="gray"))
    ax.annotate("", xy=(0, 1.6), xytext=(0, -0.2), arrowprops=dict(arrowstyle="->", color="gray"))
    ax.text(1.62, -0.07, "p (m)", color="gray")
    ax.text(0.03, 1.62, "q (m)", color="gray")

    # arrow 1: lever from O to the tip
    ax.annotate("", xy=(p, q), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="tab:blue", lw=3))
    ax.text(p / 2 - 0.02, q / 2 - 0.12, r"lever  $\vec{r}$", color="tab:blue", fontsize=11)

    # arrow 2: force starting at the tip
    ax.annotate("", xy=(p + Fp * scale, q + Fq * scale), xytext=(p, q),
                arrowprops=dict(arrowstyle="-|>", color="tab:red", lw=3))
    ax.text(p + Fp * scale + 0.03, q + Fq * scale, r"force  $\vec{F}$", color="tab:red", fontsize=11)
    ax.plot([p], [q], "ko")

    # dashed line through the tip, parallel to the p axis (the same ruler, copied to the tip)
    ax.plot([p, p + 0.7], [q, q], "k--", lw=1, alpha=0.5)

    # angle phi_r at the origin, measured from the p axis
    if phi_r > 1:
        ax.add_patch(Arc((0, 0), 0.5, 0.5, theta1=0, theta2=phi_r, color="tab:blue", lw=2))
        ax.text(0.3, 0.06, r"$\varphi_r$", color="tab:blue", fontsize=12)
    else:
        ax.text(0.3, 0.06, r"$\varphi_r = 0^\circ$", color="tab:blue", fontsize=12)

    # angle phi_F at the tip, measured from the dashed line
    ax.add_patch(Arc((p, q), 0.5, 0.5, theta1=0, theta2=phi_F, color="tab:red", lw=2))
    ax.text(p + 0.28, q + 0.07, r"$\varphi_F$", color="tab:red", fontsize=12)

    ax.set_title(title, fontsize=11)
    ax.set_xlim(-0.3, 1.9)
    ax.set_ylim(-0.3, 2.3)
    ax.set_aspect("equal")
    ax.axis("off")

    info = (rf"$\rho={rho:.2f}$ m,  $\varphi_r={phi_r:.1f}^\circ$" "\n"
            rf"$f={f:.1f}$ N,  $\varphi_F={phi_F:.1f}^\circ$" "\n"
            rf"$\theta=\varphi_F-\varphi_r={phi_F - phi_r:.1f}^\circ$" "\n"
            rf"$\tau=pF_q-qF_p={tau:.2f}$ N·m")
    ax.text(-0.25, 2.25, info, va="top", fontsize=10,
            bbox=dict(boxstyle="round", fc="white", ec="gray"))
    return rho, f, phi_r, phi_F, tau


fig, axes = plt.subplots(1, 2, figsize=(12, 6.4))

# case 1: rod to (1, 0) m, push 10 N straight up (+q)
r1, F1 = (1.0, 0.0), (0.0, 10.0)
out1 = draw(axes[0], r1, F1, 0.08, "Rod to (1, 0) m, push up")

# case 2: tip at (0.8, 0.6) m, push 10 N at 70 degrees from the p axis
ang = np.radians(70)
r2, F2 = (0.8, 0.6), (10 * np.cos(ang), 10 * np.sin(ang))
out2 = draw(axes[1], r2, F2, 0.08, "General case: both angles nonzero")

fig.suptitle("Two arrows on one flat page: lever (blue) and force (red)", fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(RESULT_DIR, f"t2_two_arrows_{TIMESTAMP}.png"), dpi=150)
print("saved the figure")
for name, (rho, f, phi_r, phi_F, tau) in [("case 1", out1), ("case 2", out2)]:
    print(f"{name}: rho = {rho:.3f} m, phi_r = {phi_r:.2f} deg, f = {f:.2f} N, phi_F = {phi_F:.2f} deg, "
          f"theta = {phi_F - phi_r:.2f} deg, tau = {tau:.3f} N*m, "
          f"check rho f sin(theta) = {rho * f * np.sin(np.radians(phi_F - phi_r)):.3f} N*m")
plt.show()

saved the figure
case 1: rho = 1.000 m, phi_r = 0.00 deg, f = 10.00 N, phi_F = 90.00 deg, theta = 90.00 deg, tau = 10.000 N*m, check rho f sin(theta) = 10.000 N*m
case 2: rho = 1.000 m, phi_r = 36.87 deg, f = 10.00 N, phi_F = 70.00 deg, theta = 33.13 deg, tau = 5.465 N*m, check rho f sin(theta) = 5.465 N*m


/var/folders/dh/551p960n6mv9t9byzj8pp0640000gp/T/ipykernel_34199/1555722818.py:100: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
